In [12]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output
import re

# We will use pandas to load and manipulate the dataset
# and ipywidgets to create our manual review UI.

In [13]:
# 1. Load the parquet file
file_path = "/home/dilmurod/job/tts_project/data/cleaned/karakalpak_speech_corpus/parquet_files/train-00000-of-00008.parquet"
df = pd.read_parquet(file_path)

# 2. Find all rows where the normalized text contains "bútin bes"
mask = df['text'].astype(str).str.contains('bútin bes', na=False)
target_indices = df[mask].index.tolist()

print(f"Found {len(target_indices)} audio files requiring manual review for '.5'.")

Found 43 audio files requiring manual review for '.5'.


In [15]:
# Create interactive widgets
out = widgets.Output()
text_input = widgets.Textarea(layout=widgets.Layout(width='80%', height='80px'))
btn_next = widgets.Button(description="Save & Next", button_style='success')
status_label = widgets.Label()

current_pos = 0

def show_current_row():
    with out:
        clear_output(wait=True)
        
        if current_pos >= len(target_indices):
            print("🎉 All done! You have reviewed all targeted rows.")
            print("Run the next cell to save your changes to the Parquet file.")
            text_input.disabled = True
            btn_next.disabled = True
            return
            
        idx = target_indices[current_pos]
        row = df.loc[idx]
        
        status_label.value = f"Reviewing {current_pos + 1} / {len(target_indices)} | ID: {row['id']}"
        print(f"Raw Text: {row.get('raw_text', 'N/A')}")
        
        # Smart audio player (handles raw bytes, file paths, or HuggingFace array formats)
        audio_data = row['cleaned_audio']
        if isinstance(audio_data, dict):
            if 'bytes' in audio_data and audio_data['bytes'] is not None:
                display(Audio(data=audio_data['bytes']))
            elif 'array' in audio_data and 'sampling_rate' in audio_data:
                display(Audio(data=audio_data['array'], rate=audio_data['sampling_rate']))
            elif 'path' in audio_data and audio_data['path']:
                display(Audio(filename=audio_data['path']))
        elif isinstance(audio_data, str):
            display(Audio(filename=audio_data))
        else:
            print("⚠️ Cannot auto-play audio format. Audio column data:")
            print(audio_data)
            
        # Populate the text box with the currently normalized text
        text_input.value = row['text']

def on_save_and_next(b):
    global current_pos
    if current_pos < len(target_indices):
        idx = target_indices[current_pos]
        
        # Save the edited text back into the dataframe
        df.at[idx, 'text'] = text_input.value
        
        # Move forward
        current_pos += 1
        show_current_row()

# Link the button to the function
btn_next.on_click(on_save_and_next)

# Display the UI
display(status_label, out, text_input, btn_next)
show_current_row()

Label(value='')

Output()

Textarea(value='', layout=Layout(height='80px', width='80%'))

Button(button_style='success', description='Save & Next', style=ButtonStyle())

In [4]:
import pyarrow as pa
import pyarrow.parquet as pq

# 1. Read the exact original strict schema from the file
original_schema = pq.read_schema(file_path)

# 2. Convert the Pandas DataFrame back into a PyArrow Table, 
# strictly enforcing the original schema (preventing 'large_string' mutations)
table = pa.Table.from_pandas(df, schema=original_schema)

# 3. Write it back to disk using PyArrow directly, bypassing Pandas entirely
pq.write_table(table, file_path)

print(f"Success! Overwrote {file_path} with manual corrections while perfectly preserving the schema.")

Success! Overwrote /home/dilmurod/job/tts_project/data/cleaned/karakalpak_speech_corpus/parquet_files/train-00000-of-00008.parquet with your manual corrections.


In [32]:
# checking audio from original source


from IPython.display import Audio, display

target_id = 14346

row = df.loc[df["id"] == target_id].iloc[0]
audio_data = row["cleaned_audio"]

if isinstance(audio_data, dict) and audio_data.get("bytes") is not None:
    display(Audio(data=audio_data["bytes"]))

In [ ]:
### number checking:

import re

PHONE_PATTERNS = {
    "uzbekistan": re.compile(
        r"""
        (?<!\d)
        (?:\+998[\s\-()]*)?
        (?:9[0-9])
        [\s\-()]*
        \d{3}
        [\s\-()]*
        \d{2}
        [\s\-()]*
        \d{2}
        (?!\d)
        """,
        re.VERBOSE
    ),

    "russia": re.compile(
        r"""
        (?<!\d)
        (?:\+7|8)
        [\s\-()]*
        (?:[3489]\d{2})
        [\s\-()]*
        \d{3}
        [\s\-()]*
        \d{2}
        [\s\-()]*
        \d{2}
        (?!\d)
        """,
        re.VERBOSE
    ),

    "usa": re.compile(
        r"""
        (?<!\d)
        (?:\+1[\s\-()]*)?
        (?:\(?[2-9]\d{2}\)?)
        [\s\-]*
        \d{3}
        [\s\-]*
        \d{4}
        (?!\d)
        """,
        re.VERBOSE
    ),
}

 loading full dataset

In [1]:
from pathlib import Path
import pandas as pd

folder = Path("/home/dilmurod/job/tts_project/data/cleaned/karakalpak_speech_corpus/parquet_files")

files = list(folder.rglob("*.parquet"))

df = pd.concat(
    [pd.read_parquet(file) for file in files],
    ignore_index=True
)

print(f"Files: {len(files)}")
print(f"Rows: {len(df)}")
df.head()

Files: 10
Rows: 21685


,id,cleaned_audio,raw_text,text
0,12500,{'bytes': b'OggS\x00\x02\x00\x00\x00\x00\x00\x...,"Belgilengen ilajlar izbe-iz ámelge asırılsa, a...",belgilengen ilajlar izbe alıw iz ámelge asırıl...
1,12501,{'bytes': b'OggS\x00\x02\x00\x00\x00\x00\x00\x...,"Bunnan tısqarı, Saida Mirziyoeva Vena qalasınd...",bunnan tısqarı saida mirziyoeva vena qalasında...
2,12502,{'bytes': b'OggS\x00\x02\x00\x00\x00\x00\x00\x...,Pidayı hayallarǵa húrmet 15-oktyabr – Xalıqara...,pidayı hayallarǵa húrmet on besinshi oktyabr x...
3,12503,{'bytes': b'OggS\x00\x02\x00\x00\x00\x00\x00\x...,"Ol tek ǵana málimleme deregi emes, al insan ru...",ol tek ǵana málimleme deregi emes al insan ruw...
4,12504,{'bytes': b'OggS\x00\x02\x00\x00\x00\x00\x00\x...,Olar ushın jaratılǵan huqıqıy kepillikler hám ...,olar ushın jaratılǵan huqıqıy kepillikler hám ...


In [12]:
# seeing random sample

import random
from IPython.display import Audio, display

import random as r

target_id = r.randint(1,21680)

row = df.loc[df["id"] == target_id].iloc[0]
audio_data = row["cleaned_audio"]
text=row["text"]
raw_text=row["raw_text"]

print("Text: ",text)
print("-"*50)
print("raw_text: ",raw_text)
print("Id",target_id)

if isinstance(audio_data, dict) and audio_data.get("bytes") is not None:
    display(Audio(data=audio_data["bytes"]))

Text:  usı baǵdardaǵı oqıw alıw seminarı qanlıkól rayonında da bolıp ótti onda xalıq deputatları rayonlıq keńesi deputatları sekretariat baslıǵı siyasiy partiya baslıqları qatnastı ilajdı xalıq deputatları qanlıkól rayonlıq keńesi baslıǵı b sultanov basqarıp bardı
--------------------------------------------------
raw_text:  Usı baǵdardaǵı oqıw-seminarı Qanlıkól rayonında da bolıp ótti. Onda xalıq deputatları rayonlıq Keńesi deputatları, sekretariat baslıǵı, siyasiy partiya baslıqları qatnastı. Ilajdı Xalıq deputatları Qanlıkól rayonlıq Keńesi baslıǵı B.Sultanov basqarıp bardı.
Id 5844


In [27]:
for _, row in df.iterrows():
    text = str(row["raw_text"])
    tt=str(row["text"])
    found = []

    for country, pattern in PHONE_PATTERNS.items():
        matches = pattern.findall(text)

        if matches:
            found.append((country, matches))

    if found:
        print(f"ID: {row['id']}")
        print(f"raw_text: {text}")
        print(f"text: {tt}")

        #for country, matches in found:
            #print(f"  {country}: {matches}")

        print("-" * 60)

ID: 14346
raw_text: Usı maqsette 9 arnawlı otryad shólkemlestirilip, zárúr texnikalar hám qurallar tolıq tayar jaǵdayǵa keltirilgen. Seminarda, eger shegirtkeniń shaǵıwı yamasa zıyan keltiriwi menen baylanıslı jaǵdayǵa dus kelse oraylıq shtabqa (97) 651-53-53 yamasa aymaqqa biriktirilgen otryad nomerleri ( ne baylanısıwı múmkin ekenligi tuwralı maǵlıwmatlar berilip, ximiyalıq islew beriw hám xızmetler biypul ekenligi atap ótildi.
text: usı maqsette toǵız arnawlı otryad shólkemlestirilip zárúr texnikalar hám qurallar tolıq tayar jaǵdayǵa keltirilgen seminarda eger shegirtkeniń shaǵıwı yamasa zıyan keltiriwi menen baylanıslı jaǵdayǵa dus kelse oraylıq shtabqa toqsan jeti altı júz eliw bir eliw úsh alıw eliw úsh yamasa aymaqqa biriktirilgen otryad nomerleri ne baylanısıwı múmkin ekenligi tuwralı maǵlıwmatlar berilip ximiyalıq islew beriw hám xızmetler biypul ekenligi atap ótildi
------------------------------------------------------------
ID: 5401
raw_text: Biraq, puqaralardan eger shegir

In [24]:
phone_rows = []

for _, row in df.iterrows():
    text = str(row["raw_text"])

    countries = []

    for country, pattern in PHONE_PATTERNS.items():
        if pattern.search(text):
            countries.append(country)

    if countries:
        phone_rows.append({
            "id": row["id"],
            "countries": countries,
            "text": text,
        })

phone_rows

[{'id': 14346,
  'countries': ['uzbekistan'],
  'text': 'Usı maqsette 9 arnawlı otryad shólkemlestirilip, zárúr texnikalar hám qurallar tolıq tayar jaǵdayǵa keltirilgen. Seminarda, eger shegirtkeniń shaǵıwı yamasa zıyan keltiriwi menen baylanıslı jaǵdayǵa dus kelse oraylıq shtabqa (97) 651-53-53 yamasa aymaqqa biriktirilgen otryad nomerleri ( ne baylanısıwı múmkin ekenligi tuwralı maǵlıwmatlar berilip, ximiyalıq islew beriw hám xızmetler biypul ekenligi atap ótildi.'},
 {'id': 5401,
  'countries': ['uzbekistan'],
  'text': 'Biraq, puqaralardan eger shegirtke shabılıwına dus kelse, olar operativ shtabtıń 97-651-53-53 telefon nomerine xabarlasıwın sorap qalar edik. Qánigelerimiz tezlik penen jaǵdaydı úyrenedi hám ximiyalıq qayta islew beriw ilajların ámelge asıradı.'}]